# 💼 Notebook 04 — Business Insights

**Project:** Churn Platform  
**Audience:** Business Stakeholders, Product Managers, Marketing  
**Goal:** Translate model outputs into revenue impact, customer segmentation, and actionable retention recommendations.

---
### Sections
1. Environment Setup
2. Load Data & Model Predictions
3. KPI Dashboard — Churn Health Metrics
4. Revenue at Risk Analysis
5. Customer Segmentation (Risk Tiers)
6. Churn Driver Deep-Dive
7. Retention ROI Estimation
8. Model Value — Lift & Gain Charts
9. Strategic Recommendations

## 1. Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import json
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi'   : 120,
    'figure.figsize': (13, 5),
    'axes.titlesize': 13,
})

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../models')
RANDOM_STATE  = 42

# ── Brand color palette ───────────────────────────────────────────────────────
C_RED   = '#F44336'
C_GREEN = '#4CAF50'
C_BLUE  = '#2196F3'
C_ORG   = '#FF9800'
C_PURP  = '#9C27B0'

print('Environment ready.')

## 2. Load Data & Model Predictions

In [ ]:
# ── Raw data (for business-readable fields) ───────────────────────────────────
df_raw = pd.read_csv('../data/raw_data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')
df_raw['Churn']        = df_raw['Churn'].map({'Yes': 1, 'No': 0})
df_raw['TotalCharges'].fillna(df_raw['MonthlyCharges'], inplace=True)

# ── Processed feature matrix ──────────────────────────────────────────────────
X = pd.read_csv(PROCESSED_DIR / 'X.csv')
y = pd.read_csv(PROCESSED_DIR / 'y.csv').squeeze()

# ── Load best model ───────────────────────────────────────────────────────────
model = joblib.load(MODELS_DIR / 'best_model.pkl')
with open(MODELS_DIR / 'model_metadata.json') as f:
    meta = json.load(f)

THRESHOLD = meta.get('decision_threshold', 0.5)

# ── Generate predictions for full dataset (train+test) ───────────────────────
y_prob = model.predict_proba(X)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

# Attach predictions to raw df (align on index)
df_raw = df_raw.reset_index(drop=True)
df_raw['churn_prob'] = y_prob
df_raw['churn_pred'] = y_pred

print(f'Dataset: {df_raw.shape}')
print(f'Model threshold: {THRESHOLD}')
print(f'Predicted churners: {y_pred.sum():,} ({y_pred.mean()*100:.1f}%)')

## 3. KPI Dashboard — Churn Health Metrics

In [ ]:
# ── Key Business Metrics ──────────────────────────────────────────────────────
total_customers   = len(df_raw)
actual_churners   = df_raw['Churn'].sum()
actual_churn_rate = df_raw['Churn'].mean()

total_mrr         = df_raw['MonthlyCharges'].sum()
mrr_at_risk       = df_raw[df_raw['Churn'] == 1]['MonthlyCharges'].sum()
avg_ltv           = df_raw['TotalCharges'].mean()
avg_tenure        = df_raw['tenure'].mean()

# Monthly churn rate (from telco context: typically monthly)
monthly_churn_rate = actual_churn_rate  # 1-period snapshot

# Customer Lifetime Value proxy: avg monthly charge × avg tenure
avg_clv = df_raw['MonthlyCharges'].mean() * df_raw['tenure'].mean()

# Revenue lost to churn (TotalCharges of churned customers)
revenue_lost = df_raw[df_raw['Churn'] == 1]['TotalCharges'].sum()

kpis = [
    ('Total Customers',       f'{total_customers:,}',          C_BLUE),
    ('Actual Churners',       f'{actual_churners:,}',          C_RED),
    ('Churn Rate',            f'{actual_churn_rate*100:.1f}%', C_ORG),
    ('Total MRR',             f'${total_mrr:,.0f}',            C_GREEN),
    ('MRR at Risk (churners)',f'${mrr_at_risk:,.0f}',          C_RED),
    ('Avg Customer LTV',      f'${avg_clv:,.0f}',              C_PURP),
    ('Avg Tenure',            f'{avg_tenure:.1f} mo',          C_BLUE),
    ('Revenue Lost',          f'${revenue_lost:,.0f}',         C_RED),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 6))
axes = axes.flatten()

for i, (label, value, color) in enumerate(kpis):
    axes[i].set_facecolor(color + '18')  # light tint
    axes[i].text(0.5, 0.62, value,  transform=axes[i].transAxes,
                 ha='center', va='center', fontsize=22, fontweight='bold', color=color)
    axes[i].text(0.5, 0.25, label,  transform=axes[i].transAxes,
                 ha='center', va='center', fontsize=11, color='#444')
    for spine in axes[i].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
    axes[i].set_xticks([])
    axes[i].set_yticks([])

plt.suptitle('📊 Churn Platform — KPI Dashboard', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Revenue at Risk Analysis

In [ ]:
# ── Monthly charges distribution — churned vs retained ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# KDE: monthly charges
for churn_val, label, color in [(0, 'Retained', C_BLUE), (1, 'Churned', C_RED)]:
    data = df_raw[df_raw['Churn'] == churn_val]['MonthlyCharges']
    data.plot.kde(ax=axes[0], color=color, linewidth=2.5, label=label)
    axes[0].axvline(data.median(), color=color, linestyle='--', linewidth=1.2, alpha=0.7)

axes[0].set_title('Monthly Charges Distribution\n(Churned vs Retained)', fontweight='bold')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].legend()

# Revenue at risk by contract type
rev_risk = df_raw[df_raw['Churn'] == 1].groupby('Contract')['MonthlyCharges'].agg(['sum', 'count'])
rev_risk.columns = ['MRR_at_Risk', 'Churned_Count']
rev_risk = rev_risk.sort_values('MRR_at_Risk', ascending=False)

bars = axes[1].bar(rev_risk.index, rev_risk['MRR_at_Risk'],
                   color=[C_RED, C_ORG, C_GREEN], edgecolor='white')
ax2 = axes[1].twinx()
ax2.plot(rev_risk.index, rev_risk['Churned_Count'],
         'D-', color='navy', linewidth=2, markersize=8, label='Churned count')

axes[1].set_title('MRR at Risk by Contract Type', fontweight='bold')
axes[1].set_ylabel('MRR at Risk ($)')
ax2.set_ylabel('Number of Churned Customers')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

for bar, val in zip(bars, rev_risk['MRR_at_Risk']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'${val:,.0f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print('Revenue at Risk by Contract:')
print(rev_risk.assign(MRR_at_Risk=rev_risk['MRR_at_Risk'].map('${:,.2f}'.format)).to_string())

In [ ]:
# ── Revenue at risk by Internet Service ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for col, ax, title in [
    ('InternetService', axes[0], 'Internet Service'),
    ('PaymentMethod',   axes[1], 'Payment Method')
]:
    seg = df_raw[df_raw['Churn'] == 1].groupby(col)['MonthlyCharges'].sum().sort_values(ascending=False)
    colors_seg = sns.color_palette('Reds_r', len(seg))
    bars = ax.bar(seg.index.astype(str), seg.values, color=colors_seg, edgecolor='white')
    ax.set_title(f'MRR at Risk by {title}', fontweight='bold')
    ax.set_ylabel('MRR at Risk ($)')
    ax.tick_params(axis='x', rotation=20)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    for bar, val in zip(bars, seg.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'${val:,.0f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Customer Segmentation (Risk Tiers)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Segment customers into 4 risk tiers based on predicted probability
#   🟢 Low Risk     : prob < 0.25
#   🟡 Medium Risk  : 0.25 ≤ prob < 0.50
#   🟠 High Risk    : 0.50 ≤ prob < 0.75
#   🔴 Critical Risk: prob ≥ 0.75
# ──────────────────────────────────────────────────────────────────────────────

def assign_risk_tier(prob):
    if prob < 0.25:   return '🟢 Low Risk'
    elif prob < 0.50: return '🟡 Medium Risk'
    elif prob < 0.75: return '🟠 High Risk'
    else:             return '🔴 Critical Risk'

df_raw['risk_tier'] = df_raw['churn_prob'].apply(assign_risk_tier)

tier_order  = ['🟢 Low Risk', '🟡 Medium Risk', '🟠 High Risk', '🔴 Critical Risk']
tier_colors = [C_GREEN, '#FFC107', C_ORG, C_RED]

tier_summary = df_raw.groupby('risk_tier').agg(
    Customers         = ('customerID', 'count'),
    Avg_Churn_Prob    = ('churn_prob', 'mean'),
    Actual_Churn_Rate = ('Churn', 'mean'),
    MRR_at_Risk       = ('MonthlyCharges', 'sum'),
    Avg_Monthly_Charge= ('MonthlyCharges', 'mean'),
    Avg_Tenure        = ('tenure', 'mean')
).reindex(tier_order)

tier_summary['% of Customers']  = (tier_summary['Customers'] / len(df_raw) * 100).round(1)
tier_summary['Actual_Churn_Rate'] = (tier_summary['Actual_Churn_Rate'] * 100).round(1)
tier_summary['Avg_Churn_Prob']  = (tier_summary['Avg_Churn_Prob'] * 100).round(1)

print('── Customer Risk Tier Segmentation ──')
print(tier_summary[['Customers', '% of Customers', 'Avg_Churn_Prob',
                     'Actual_Churn_Rate', 'MRR_at_Risk', 'Avg_Monthly_Charge', 'Avg_Tenure']]
      .rename(columns={
          'Avg_Churn_Prob'   : 'Avg Predicted Prob (%)',
          'Actual_Churn_Rate': 'Actual Churn (%)',
          'MRR_at_Risk'      : 'MRR ($)',
          'Avg_Monthly_Charge': 'Avg MRR/customer',
          'Avg_Tenure'       : 'Avg Tenure (mo)'
      }).to_string())

In [ ]:
# ── Visualize segments ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Customer count
axes[0].bar(
    tier_order,
    tier_summary.loc[tier_order, 'Customers'],
    color=tier_colors, edgecolor='white'
)
axes[0].set_title('Customer Count by Risk Tier', fontweight='bold')
axes[0].set_ylabel('Customers')
axes[0].tick_params(axis='x', rotation=15)

# MRR at risk
axes[1].bar(
    tier_order,
    tier_summary.loc[tier_order, 'MRR_at_Risk'],
    color=tier_colors, edgecolor='white'
)
axes[1].set_title('Total MRR by Risk Tier', fontweight='bold')
axes[1].set_ylabel('Monthly Recurring Revenue ($)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=15)

# Probability distribution — violin
tier_data = [df_raw[df_raw['risk_tier'] == t]['churn_prob'].values for t in tier_order]
vp = axes[2].violinplot(tier_data, showmedians=True)
for body, color in zip(vp['bodies'], tier_colors):
    body.set_facecolor(color)
    body.set_alpha(0.7)
axes[2].set_xticks(range(1, 5))
axes[2].set_xticklabels(tier_order, rotation=12)
axes[2].set_title('Predicted Probability by Tier', fontweight='bold')
axes[2].set_ylabel('Churn Probability')
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))

plt.suptitle('Customer Risk Tier Segmentation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Churn Driver Deep-Dive

In [ ]:
# ── Churn rate heatmap — Contract × Internet Service ─────────────────────────
pivot1 = df_raw.pivot_table(values='Churn', index='Contract',
                             columns='InternetService', aggfunc='mean') * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(
    pivot1, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, ax=axes[0],
    annot_kws={'size': 12, 'weight': 'bold'}
)
axes[0].set_title('Churn Rate (%) — Contract × Internet Service', fontweight='bold')

# Churn rate — Contract × Payment Method
pivot2 = df_raw.pivot_table(values='Churn', index='Contract',
                              columns='PaymentMethod', aggfunc='mean') * 100
sns.heatmap(
    pivot2, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, ax=axes[1],
    annot_kws={'size': 11, 'weight': 'bold'}
)
axes[1].set_title('Churn Rate (%) — Contract × Payment Method', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# ── Churn rate by add-on services ─────────────────────────────────────────────
SERVICE_COLS = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

service_churn = []
for col in SERVICE_COLS:
    for val in df_raw[col].unique():
        mask = df_raw[col] == val
        service_churn.append({
            'Service': col,
            'Status' : val,
            'Churn_Rate': df_raw.loc[mask, 'Churn'].mean() * 100,
            'Count': mask.sum()
        })

sc_df = pd.DataFrame(service_churn)

fig, ax = plt.subplots(figsize=(14, 6))
pivot_sc = sc_df.pivot(index='Service', columns='Status', values='Churn_Rate')

# Keep only Yes/No for clarity (drop 'No internet service' etc.)
pivot_sc = pivot_sc[['No', 'Yes']] if 'Yes' in pivot_sc.columns else pivot_sc
pivot_sc.plot(kind='bar', ax=ax,
              color=[C_BLUE, C_RED], edgecolor='white', width=0.6)

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.axhline(df_raw['Churn'].mean() * 100, color='black',
           linestyle='--', linewidth=1.2, label=f'Overall avg ({df_raw["Churn"].mean()*100:.1f}%)')
ax.set_title('Churn Rate by Add-On Service Subscription', fontweight='bold')
ax.set_xlabel('Service')
ax.set_ylabel('Churn Rate (%)')
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Subscribed')
plt.tight_layout()
plt.show()

In [ ]:
# ── Churn probability vs tenure & monthly charges (scatter) ──────────────────
sample_df = df_raw.sample(min(2000, len(df_raw)), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(14, 7))
scatter = ax.scatter(
    sample_df['tenure'],
    sample_df['MonthlyCharges'],
    c=sample_df['churn_prob'],
    cmap='RdYlGn_r',
    alpha=0.7,
    s=40,
    edgecolors='white',
    linewidths=0.3
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Predicted Churn Probability', fontsize=11)
cbar.ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Monthly Charges ($)')
ax.set_title('Customer Risk Map — Tenure vs Monthly Charges\n(color = predicted churn probability)',
             fontweight='bold')

# Annotate danger zone
ax.axvline(12, color='black', linewidth=1, linestyle=':', alpha=0.6)
ax.text(13, ax.get_ylim()[1]*0.93, '← Early stage (high risk)', fontsize=9, color='#333')
plt.tight_layout()
plt.show()

## 7. Retention ROI Estimation

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Assumptions (adjust to real business numbers):
#   • Avg retention intervention cost per customer = $50
#   • Success rate of retention campaign = 30%
#   • Avg customer LTV = avg_monthly_charges × avg_remaining_tenure
#   • Targeting CRITICAL + HIGH risk tiers only (focused spend)
# ──────────────────────────────────────────────────────────────────────────────

INTERVENTION_COST     = 50    # $/customer
RETENTION_SUCCESS_RATE = 0.30  # 30% of targeted customers are retained
MONTHS_AHEAD          = 12    # planning horizon

# Identify high & critical risk customers
target_df = df_raw[df_raw['risk_tier'].isin(['🟠 High Risk', '🔴 Critical Risk'])].copy()

# Estimated LTV saved per retained customer
target_df['expected_remaining_months'] = MONTHS_AHEAD  # simplified
target_df['ltv_at_risk']               = target_df['MonthlyCharges'] * target_df['expected_remaining_months']

# ROI calculation
n_targeted         = len(target_df)
campaign_cost      = n_targeted * INTERVENTION_COST
n_retained         = int(n_targeted * RETENTION_SUCCESS_RATE)
revenue_saved      = target_df.nlargest(n_retained, 'ltv_at_risk')['ltv_at_risk'].sum()
net_gain           = revenue_saved - campaign_cost
roi_pct            = (net_gain / campaign_cost) * 100 if campaign_cost > 0 else 0

print(f"""
╔════════════════════════════════════════════════════════════╗
║              RETENTION CAMPAIGN ROI ESTIMATE               ║
╚════════════════════════════════════════════════════════════╝

Assumptions:
  • Targeting: High Risk + Critical Risk customers only
  • Intervention cost : ${INTERVENTION_COST}/customer
  • Retention success : {RETENTION_SUCCESS_RATE*100:.0f}%
  • Planning horizon  : {MONTHS_AHEAD} months

Results:
  Customers targeted   : {n_targeted:,}
  Campaign cost        : ${campaign_cost:,.0f}
  Estimated saved cust : {n_retained:,}
  Revenue saved (LTV)  : ${revenue_saved:,.0f}
  Net gain             : ${net_gain:,.0f}
  ROI                  : {roi_pct:.1f}%
""")

In [ ]:
# ── Sensitivity analysis — ROI vs intervention cost ──────────────────────────
costs   = np.arange(10, 201, 10)
rois    = []
net_gains = []

for cost in costs:
    c_cost   = n_targeted * cost
    n_ret    = int(n_targeted * RETENTION_SUCCESS_RATE)
    rev_sav  = target_df.nlargest(n_ret, 'ltv_at_risk')['ltv_at_risk'].sum()
    ng       = rev_sav - c_cost
    net_gains.append(ng)
    rois.append((ng / c_cost) * 100 if c_cost > 0 else 0)

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.fill_between(costs, net_gains, alpha=0.2, color=C_GREEN)
ax1.plot(costs, net_gains, color=C_GREEN, linewidth=2.5, label='Net Gain ($)')
ax1.set_ylabel('Net Gain ($)', color=C_GREEN)
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.axhline(0, color='black', linewidth=0.8)

ax2.plot(costs, rois, color=C_BLUE, linewidth=2.5, linestyle='--', marker='o', markersize=4, label='ROI (%)')
ax2.set_ylabel('ROI (%)', color=C_BLUE)

ax1.set_xlabel('Intervention Cost per Customer ($)')
ax1.set_title('ROI Sensitivity Analysis — Intervention Cost vs Revenue Saved', fontweight='bold')
ax1.axvline(INTERVENTION_COST, color=C_RED, linewidth=1.5, linestyle=':', label=f'Current assumption (${INTERVENTION_COST})')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

## 8. Model Value — Lift & Gain Charts

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Lift & Gain charts show how much better the model is vs random targeting.
#   • Gain  = % of actual churners captured if we target top X% of customers
#   • Lift  = Gain / (X% baseline)
# ──────────────────────────────────────────────────────────────────────────────

# Sort by predicted probability descending
lift_df = df_raw[['Churn', 'churn_prob']].sort_values('churn_prob', ascending=False).reset_index(drop=True)

n          = len(lift_df)
n_churners = lift_df['Churn'].sum()

lift_df['cumulative_churners'] = lift_df['Churn'].cumsum()
lift_df['pct_customers']       = (lift_df.index + 1) / n * 100
lift_df['cumulative_gain']     = lift_df['cumulative_churners'] / n_churners * 100
lift_df['lift']                = lift_df['cumulative_gain'] / lift_df['pct_customers']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cumulative Gain Chart
axes[0].plot(lift_df['pct_customers'], lift_df['cumulative_gain'],
             color=C_RED, linewidth=2.5, label='XGBoost Model')
axes[0].plot([0, 100], [0, 100], 'k--', linewidth=1.2, label='Random (baseline)')
axes[0].plot([0, n_churners/n*100, 100], [0, 100, 100],
             color=C_GREEN, linewidth=1.5, linestyle=':', label='Perfect model')
axes[0].fill_between(lift_df['pct_customers'],
                      lift_df['cumulative_gain'],
                      lift_df['pct_customers'], alpha=0.1, color=C_RED)
axes[0].set_xlabel('% of Customers Targeted')
axes[0].set_ylabel('% of Churners Captured')
axes[0].set_title('Cumulative Gain Chart', fontweight='bold')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# Lift Chart
axes[1].plot(lift_df['pct_customers'], lift_df['lift'],
             color=C_BLUE, linewidth=2.5, label='XGBoost Model')
axes[1].axhline(1, color='black', linewidth=1.2, linestyle='--', label='Random (lift = 1)')
axes[1].set_xlabel('% of Customers Targeted')
axes[1].set_ylabel('Lift')
axes[1].set_title('Lift Chart', fontweight='bold')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_ylim(0, lift_df['lift'].quantile(0.98) * 1.1)

plt.suptitle('Model Value — Lift & Gain Charts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Report lift at key deciles
print('── Lift at Key Deciles ──')
for decile in [10, 20, 30, 50]:
    row = lift_df[lift_df['pct_customers'] <= decile].iloc[-1]
    print(f'  Top {decile}% customers → Gain: {row["cumulative_gain"]:.1f}%  |  Lift: {row["lift"]:.2f}x')

## 9. Strategic Recommendations

In [ ]:
# ── Supporting data for recommendations ───────────────────────────────────────
mtm_churn  = df_raw[df_raw['Contract'] == 'Month-to-month']['Churn'].mean() * 100
twyr_churn = df_raw[df_raw['Contract'] == 'Two year']['Churn'].mean() * 100
fiber_churn= df_raw[df_raw['InternetService'] == 'Fiber optic']['Churn'].mean() * 100
nosec_churn= df_raw[df_raw['OnlineSecurity'] == 'No']['Churn'].mean() * 100
echeck_ch  = df_raw[df_raw['PaymentMethod'] == 'Electronic check']['Churn'].mean() * 100
early_churn= df_raw[df_raw['tenure'] <= 12]['Churn'].mean() * 100

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║             📋 STRATEGIC CHURN REDUCTION RECOMMENDATIONS             ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. 📄 CONTRACT UPGRADES  [Highest Impact]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Month-to-month churn : {mtm_churn:.1f}%  vs  Two-year churn: {twyr_churn:.1f}%
   → Offer discounts/loyalty bonuses for month-to-month customers
     to upgrade to annual or bi-annual plans.
   → Introduce a "lock-in incentive": waive first month's fee on
     upgrade to 1- or 2-year contract.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. ⏱️  EARLY TENURE ONBOARDING  [Critical Window]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Churn rate in first 12 months: {early_churn:.1f}%
   → Implement a structured onboarding program for new customers.
   → Send proactive check-ins at months 1, 3, and 6.
   → Offer service credits during the first 3 months.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. 🛡️  SECURITY & SUPPORT ADD-ONS  [Engagement Driver]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Customers without OnlineSecurity churn at {nosec_churn:.1f}%
   → Bundle OnlineSecurity + TechSupport as a free 3-month trial
     for fiber optic customers (fiber churn: {fiber_churn:.1f}%).
   → Highlight security value in onboarding communications.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
4. 💳 PAYMENT METHOD MIGRATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Electronic check customers churn at {echeck_ch:.1f}%
   → Incentivize migration to auto-pay (credit card or bank transfer)
     with a monthly bill discount ($5-$10/month).
   → Auto-pay reduces involuntary churn (failed payments).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5. 🤖 MODEL-DRIVEN OUTREACH  [Operational]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   → Deploy the XGBoost model via the Churn Platform API.
   → Score all customers weekly; route Critical (prob≥0.75) and
     High risk (0.50-0.75) to retention specialists.
   → Use the Lift Chart: targeting only top 20% customers captures
     ~50%+ of all churners — focus resources efficiently.
   → Re-score monthly; track tier movements as health KPI.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
6. 📊 MONITORING & FEEDBACK LOOP
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   → Log all predictions and actual outcomes to OLAP schema.
   → Retrain model quarterly with fresh data.
   → Track: churn rate trend, MRR saved, intervention success rate.
""")

In [ ]:
# ── Final summary chart — churn rate by key segment combos ────────────────────
segment_analysis = df_raw.groupby(['Contract', 'InternetService']).agg(
    Customers   = ('Churn', 'count'),
    Churn_Rate  = ('Churn', 'mean'),
    Avg_Charges = ('MonthlyCharges', 'mean')
).reset_index()
segment_analysis['Churn_Rate'] *= 100
segment_analysis['Segment'] = segment_analysis['Contract'] + '\n' + segment_analysis['InternetService']
segment_analysis = segment_analysis.sort_values('Churn_Rate', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors_seg = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(segment_analysis)))
bars = ax.bar(segment_analysis['Segment'], segment_analysis['Churn_Rate'],
              color=colors_seg, edgecolor='white')

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.axhline(df_raw['Churn'].mean() * 100, color='navy', linewidth=1.5,
           linestyle='--', label=f'Avg churn ({df_raw["Churn"].mean()*100:.1f}%)')
ax.set_title('Churn Rate by Segment (Contract × Internet Service)', fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.tick_params(axis='x', rotation=15)
ax.legend()

# Add customer count as text
for bar, (_, row) in zip(bars, segment_analysis.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'n={row.Customers:,}', ha='center', fontsize=8, color='#333')

plt.tight_layout()
plt.show()

print('\n── Top 3 Highest-Risk Segments (prime retention targets) ──')
print(segment_analysis[['Contract', 'InternetService', 'Customers', 'Churn_Rate', 'Avg_Charges']]
      .head(3).rename(columns={'Churn_Rate': 'Churn Rate (%)', 'Avg_Charges': 'Avg MRR ($)'})
      .to_string(index=False))

In [ ]:
print('\n✅ Business Insights notebook complete.')
print('Dashboard (Streamlit) and API (FastAPI) are the next deployment steps.')